# Experiment 15 — NoSQL Database Operations Using MongoDB and Python

**Aim:** Demonstrate document insertion, querying, updating, and deletion through the PyMongo API.

**Libraries:** Python, PyMongo, mongomock, and pandas.

**Installation:** Install with `python -m pip install pymongo==4.19.0 mongomock==4.3.0`.

**Theory:** MongoDB stores BSON documents in collections. CRUD denotes create, read, update, and delete. Schema flexibility does not eliminate the need for validation, indexes, or access control.

**Execution note:** This notebook uses `mongomock` for deterministic, in-memory execution without connecting to Atlas or a real server. It demonstrates PyMongo-style API operations but is not a substitute for verifying deployment-specific behavior against MongoDB. Never put a connection URI or credentials in notebook source.

**Procedure / Implementation:** Create fictional movie documents, query with projection/sort, update one known test document, delete one known test document, and verify operation counts.

## Procedure / Implementation — Create and read sample documents

In [1]:
import pandas as pd
import mongomock

client = mongomock.MongoClient()
collection = client["lab_practical"]["movies"]
documents = [
    {"title": "Sample Film A", "year": 2022, "genres": ["Drama"], "rating": 7.5},
    {"title": "Sample Film B", "year": 2023, "genres": ["Comedy"], "rating": 6.8},
    {"title": "Sample Film C", "year": 2024, "genres": ["Drama", "Mystery"], "rating": 8.1},
]
insert_result = collection.insert_many(documents)
print("Inserted documents:", len(insert_result.inserted_ids))
print("Matching drama documents:")
for document in collection.find(
    {"genres": "Drama"},
    {"_id": 0, "title": 1, "year": 1, "rating": 1},
).sort("year", 1):
    print(document)

Inserted documents: 3
Matching drama documents:
{'title': 'Sample Film A', 'year': 2022, 'rating': 7.5}
{'title': 'Sample Film C', 'year': 2024, 'rating': 8.1}


## 2. Update, delete, and verify counts

In [2]:
from pathlib import Path

update_result = collection.update_one(
    {"title": "Sample Film B"},
    {"$set": {"rating": 7.0}},
)
updated_document = collection.find_one(
    {"title": "Sample Film B"},
    {"_id": 0},
)
delete_result = collection.delete_one({"title": "Sample Film A"})

operation_summary = pd.DataFrame(
    [
        ("inserted", len(insert_result.inserted_ids)),
        ("matched for update", update_result.matched_count),
        ("modified", update_result.modified_count),
        ("deleted", delete_result.deleted_count),
        ("remaining", collection.count_documents({})),
    ],
    columns=["operation", "count"],
)
print("Updated document:", updated_document)
display(operation_summary)
assert update_result.matched_count == 1
assert delete_result.deleted_count == 1
assert collection.count_documents({}) == 2
operation_summary.to_csv(
    Path("outputs") / "results" / "exp15_mongodb_crud.csv",
    index=False,
)
client.close()

Updated document: {'title': 'Sample Film B', 'year': 2023, 'genres': ['Comedy'], 'rating': 7.0}


,operation,count
0,inserted,3
1,matched for update,1
2,modified,1
3,deleted,1
4,remaining,2


## Observation
The executed output shows inserted, matched, modified, deleted, and remaining-document counts for fictional records.

## Result
The in-memory CRUD operations passed their assertions. The run did not connect to MongoDB Atlas or a local server; use a protected environment variable and an approved instance for a live-server practical.